pai_nosso_refatorado.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/alanabdmorais/pai_nosso_refatorado/blob/main/pai_nosso_refatorado.ipynb

# 🎬 Pipeline Multilíngue — Orações com Legendas Morfológicas

**Estratégia de classificação intermediária:**
O Groq gera as classificações → você revisa com uma IA → pipeline continua com os JSONs corrigidos.

---

### Fluxo completo

| # | Fase | O que faz | Ação |
|---|------|-----------|------|
| 0 | Setup | Instala deps, monta Drive, importa módulos | Automático |
| Init | — | Cria config, groq e pipeline | Automático |
| B0 | YouTube | Baixa legendas de referência (opcional) | Manual |
| 1 | Áudio | Edge TTS → .wav no Drive | Automático |
| 2 | Whisper | Transcrição → SRT bruto | Automático |
| 3 | Correção PT | Groq corrige o SRT | Automático |
| 4 | Traduções | Groq gera EN/ES/FR | Automático |
| **5A** | **Classificação** | **Groq classifica + gera pacote de revisão → pausa** | **Automático → pausa** |
| **5B** | **Revisão** | **Você corrige JSONs com IA externa** | **👤 Manual** |
| **5C** | **Recarregar** | **Pipeline carrega JSONs corrigidos** | **Automático** |
| 6 | Clipes | Baixa e corta vídeos do Pixabay | Automático |
| 7 | Vídeo base | Crédito + narração + trilha | Automático |
| 8 | Legendas ASS | Queima legendas coloridas no vídeo | Automático |

> **Retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`

In [37]:
# ╔══════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup (rode uma vez por sessão)    ║
# ╚══════════════════════════════════════════════════╝

# Sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp nest_asyncio
print('✅ pacotes Python')

# Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
print('✅ Drive montado')

# Copiar módulos do Drive para /content/pipeline
import shutil, os, sys, logging

PASTA_MODULOS = '/content/drive/MyDrive/pipeline/modulos'  # <-- ajuste se necessário
DESTINO       = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')
else:
    print(f'⚠️  Pasta não encontrada: {PASTA_MODULOS}')
    print('   Ajuste PASTA_MODULOS acima.')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('✅ Pronto.')

✅ ffmpeg + espeak-ng
✅ pacotes Python
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado
✅ Módulos copiados: /content/drive/MyDrive/pipeline/modulos → /content/pipeline
✅ Pronto.


In [26]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  INICIALIZAÇÃO — rode após o Setup                              ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# ── Ajuste aqui para trocar a oração ─────────────────────────────────────────
config = PipelineConfig(
    NOME_ORACAO  = 'pai_nosso',
    # Para outra oração:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',
)

groq     = GroqClient(api_key=userdata.get('GROQ_KEY'), nome_oracao=config.NOME_ORACAO)
pipeline = VideoPipeline(config, groq)

cp = Checkpoint()
print(config.resumo())
print()
print(cp.resumo())
print(f'\n▶  Próxima fase: {cp.proxima_fase_pendente() or "(tudo concluído)"}')

Oração:        pai_nosso
Áudio:         pai_nosso_audio.wav
SRT mestre:    pai_nosso_pt.srt
Vídeo final:   pai_nosso_final.mp4
Idiomas:       pt, en, es, fr
Voz Edge TTS:  pt-BR-AntonioNeural
Modelo Groq:   llama-3.3-70b-versatile
Duração clipe: 5s
Correcoes:     /content/drive/MyDrive/pipeline/correcoes/pai_nosso
Brutos:        /content/drive/MyDrive/pipeline/brutos/pai_nosso

── Checkpoint ──────────────────────────────
  audio_gerado                 ✅ concluído  [2026-06-12T10:23:49]
  srt_pt_bruto                 ✅ concluído  [2026-06-12T10:24:08]
  srt_pt_corrigido             ✅ concluído  [2026-06-12T10:24:40]
  srt_traduzidos               ⬜ pendente
  classificacoes_feitas        ✅ concluído  [2026-06-12T10:28:52]
  clipes_cortados              ⬜ pendente
  video_base_criado            ⬜ pendente
  legendas_queimadas           ⬜ pendente
────────────────────────────────────────────

▶  Próxima fase: srt_traduzidos


In [27]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAÇÃO DA ESTRUTURA DO DRIVE                           ║
# ║  (execute após a Inicialização para conferir o Drive)           ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 60)
print("📁 VERIFICANDO ESTRUTURA DO DRIVE")
print("═" * 60)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')
pasta_brutos    = Path(f'/content/drive/MyDrive/pipeline/brutos/{config.NOME_ORACAO}')

for rotulo, pasta in [("correcoes", pasta_correcoes), ("brutos", pasta_brutos)]:
    print(f"\n📁 {pasta}")
    if pasta.exists():
        arquivos = list(pasta.iterdir())
        if arquivos:
            for f in sorted(arquivos):
                print(f"   ✅ {f.name}  ({f.stat().st_size/1024:.1f} KB)")
        else:
            print("   (pasta vazia)")
    else:
        print("   ⚠️  Ainda não criada (será criada automaticamente na FASE 5A)")

print("\n" + "═" * 60)
print("💡 ESTRUTURA ESPERADA APÓS A FASE 5A:")
print(f"   MyDrive/pipeline/brutos/{config.NOME_ORACAO}/")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print(f"   MyDrive/pipeline/correcoes/{config.NOME_ORACAO}/")
print(f"   ├── prompt_revisao.md")
print(f"   ├── relatorio_classificacoes.csv")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print("═" * 60)

════════════════════════════════════════════════════════════
📁 VERIFICANDO ESTRUTURA DO DRIVE
════════════════════════════════════════════════════════════

📁 /content/drive/MyDrive/pipeline/correcoes/pai_nosso
   ✅ classificacao_pai_nosso_en.json  (5.8 KB)
   ✅ classificacao_pai_nosso_es.json  (6.2 KB)
   ✅ classificacao_pai_nosso_fr.json  (6.3 KB)
   ✅ classificacao_pai_nosso_pt.json  (5.7 KB)

📁 /content/drive/MyDrive/pipeline/brutos/pai_nosso
   ✅ classificacao_pai_nosso_en.json  (5.8 KB)
   ✅ classificacao_pai_nosso_es.json  (6.2 KB)
   ✅ classificacao_pai_nosso_fr.json  (6.3 KB)
   ✅ classificacao_pai_nosso_pt.json  (5.7 KB)

════════════════════════════════════════════════════════════
💡 ESTRUTURA ESPERADA APÓS A FASE 5A:
   MyDrive/pipeline/brutos/pai_nosso/
   └── classificacao_pai_nosso_pt/en/es/fr.json
   MyDrive/pipeline/correcoes/pai_nosso/
   ├── prompt_revisao.md
   ├── relatorio_classificacoes.csv
   └── classificacao_pai_nosso_pt/en/es/fr.json
═══════════════════════════

In [36]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA - Compatível GitHub (sem widgets)          ║
# ║  Escolha o número das opções que quer limpar                    ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil
import sys
import os

def limpeza_seletiva():
    print("═" * 65)
    print("🧹 LIMPEZA SELETIVA")
    print("═" * 65)
    print("\nEscolha o que limpar (digite os números separados por vírgula)")
    print("Exemplo: 1,3,5")
    print("═" * 65)
    print("  1 - 🎵 Áudios (.wav, .mp3)")
    print("  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]")
    print("  3 - 🎬 Vídeos gerados (base, final, clipes)")
    print("  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)")
    print("  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)")
    print("  6 - 📌 Checkpoint (checkpoint.json)")
    print("  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)")
    print("  8 - 📦 Cache Python (módulos carregados)")
    print("  9 - 🔥 LIMPAR TUDO (exceto Drive)")
    print(" 10 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)")
    print("  0 - Sair")
    print("═" * 65)

    opcoes = input("\nDigite sua escolha: ").strip()

    if opcoes == '0':
        print("Saindo...")
        return

    opcoes_lista = [int(x.strip()) for x in opcoes.split(',')]
    cont = 0

    # 1. Áudios
    if 1 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n🎵 Removendo áudios...")
        for f in Path('.').glob('*_audio.wav'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.mp3'):
            if 'musica' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 2. Legendas
    if 2 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📝 Removendo legendas...")
        for f in Path('.').glob('*.srt'):
            if 'yt_ref' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.ass'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")

    # 3. Vídeos
    if 3 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n🎬 Removendo vídeos...")
        patterns = ['*_base.mp4', '*_final.mp4', 'clipe_*.mp4', 'temp_*.mp4', 'video_*.mp4', 'raw_*.mp4']
        for pattern in patterns:
            for f in Path('.').glob(pattern):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 4. JSONs BRUTOS (pasta brutos/ no Drive)
    if 4 in opcoes_lista or 10 in opcoes_lista:
        print("\n📊 Removendo JSONs BRUTOS (pasta brutos/)...")
        pasta_brutos = Path('/content/drive/MyDrive/pipeline/brutos')
        if pasta_brutos.exists():
            for oracao_dir in pasta_brutos.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ Drive/brutos/{oracao_dir.name}/{f.name}")
        else:
            print("   ⚠️ Pasta brutos/ não encontrada")

    # 5. JSONs CORRIGIDOS (pasta correcoes/ no Drive)
    if 5 in opcoes_lista or 10 in opcoes_lista:
        print("\n✅ Removendo JSONs CORRIGIDOS (pasta correcoes/)...")
        pasta_correcoes = Path('/content/drive/MyDrive/pipeline/correcoes')
        if pasta_correcoes.exists():
            for oracao_dir in pasta_correcoes.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ Drive/correcoes/{oracao_dir.name}/{f.name}")
        else:
            print("   ⚠️ Pasta correcoes/ não encontrada")

    # 6. Checkpoint
    if 6 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📌 Removendo checkpoint...")
        cp = Path('checkpoint.json')
        if cp.exists():
            cp.unlink()
            cont += 1
            print("   🗑️ checkpoint.json")

    # 7. Pastas temporárias
    if 7 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📁 Removendo pastas temporárias...")
        pastas = ['clipes_cortados', 'clipes_prontos', 'temp_raw', '__pycache__']
        for pasta in pastas:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p)
                cont += 1
                print(f"   🗑️ {pasta}/")

    # 8. Cache Python
    if 8 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📦 Limpando cache Python...")
        modulos = ['groq_client', 'video_pipeline', 'config', 'classification',
                   'checkpoint', 'drive_utils', 'ffmpeg_utils', 'srt_utils',
                   'models', 'constants', 'pipeline']
        for m in modulos:
            if m in sys.modules:
                del sys.modules[m]
                cont += 1
                print(f"   🗑️ {m}")

    print("\n" + "═" * 65)
    print(f"✅ LIMPEZA CONCLUÍDA! {cont} item(ns) removido(s)")
    print("═" * 65)

# Executar
limpeza_seletiva()

═════════════════════════════════════════════════════════════════
🧹 LIMPEZA SELETIVA
═════════════════════════════════════════════════════════════════

Escolha o que limpar (digite os números separados por vírgula)
Exemplo: 1,3,5
═════════════════════════════════════════════════════════════════
  1 - 🎵 Áudios (.wav, .mp3)
  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]
  3 - 🎬 Vídeos gerados (base, final, clipes)
  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)
  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)
  6 - 📌 Checkpoint (checkpoint.json)
  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)
  8 - 📦 Cache Python (módulos carregados)
  9 - 🔥 LIMPAR TUDO (exceto Drive)
 10 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)
  0 - Sair
═════════════════════════════════════════════════════════════════

Digite sua escolha: 10

🎵 Removendo áudios...
   🗑️ pai_nosso_audio.wav

📝 Removendo legendas...
   🗑️ pai_nosso_es.srt
   🗑️ pai_nosso_es.es.srt
   🗑️ pai_nosso_fr.srt
   🗑️ pai_nosso_pt_edge.srt
   🗑️ 

### 🔵 Célula B0 — Opcional: baixar legendas do YouTube
Execute **antes** das fases 1–8 para ter traduções mais fiéis. Se pular, o Groq traduz a partir do PT.

In [29]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — Baixar legendas do YouTube                        ║
# ║  (BAIXA TAMBÉM A REFERÊNCIA PT para a FASE 3)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
import shutil
from drive_utils import DriveClient

cfg   = config
drive = DriveClient.get()

drive.download_se_ausente(cfg.ID_PASTA_COOKIES, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

print("═" * 60)
print("📺 BAIXANDO LEGENDAS DO YOUTUBE")
print("═" * 60)

# ── COLOQUE AS URLs DOS VÍDEOS COM LEGENDAS ABAIXO ────────────────────────────
URLS = {
    'pt': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'en': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'es': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'fr': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
}

for lang, url in URLS.items():
    nome_srt = cfg.nome_srt(lang)
    print(f'\n⬇️  Baixando {lang.upper()} de: {url[:50]}...')
    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang, '--write-auto-sub',
        '--skip-download', '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}', *cookies_flag, url
    ]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    encontrado = False
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt} salvo')
        encontrado = True
        break
    if not encontrado:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')

# Criar referência PT para a FASE 3
print("\n" + "═" * 60)
print("📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)")
print("═" * 60)
srt_pt   = Path(cfg.nome_srt('pt'))
ref_path = Path(f'{cfg.NOME_ORACAO}_yt_ref_pt.srt')
if srt_pt.exists():
    shutil.copy(srt_pt, ref_path)
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f'✅ Referência PT salva: {ref_path}  ({len(ref_legendas)} segmentos)')
    print(f'   Preview: {ref_legendas[0].texto[:80]}')
else:
    print(f'⚠️  SRT PT não encontrado. A FASE 3 corrigirá sem referência.')
print("═" * 60)

════════════════════════════════════════════════════════════
📺 BAIXANDO LEGENDAS DO YOUTUBE
════════════════════════════════════════════════════════════

⬇️  Baixando PT de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_pt.srt salvo

⬇️  Baixando EN de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_en.srt salvo

⬇️  Baixando ES de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_es.srt salvo

⬇️  Baixando FR de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_fr.srt salvo

════════════════════════════════════════════════════════════
📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)
════════════════════════════════════════════════════════════
✅ Referência PT salva: pai_nosso_yt_ref_pt.srt  (11 segmentos)
   Preview: Pai nosso que estais no  céu,
════════════════════════════════════════════════════════════


### ▶ Fases 1–4 — Automáticas

In [30]:
# ── FASE 1: Áudio com Edge TTS ───────────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')

# ── FASE 2: Transcrição Whisper ──────────────────────────────────────────────
srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
print(f'{srt_bruto.name}:')
for leg in ler_srt(srt_bruto)[:4]:
    print(f'  [{leg.inicio_str}]  {leg.texto}')

✅ pai_nosso_audio.wav  (162 KB)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


pai_nosso_pt_edge.srt:
  [00:00:00,000]  Pai nosso que estáis no céu, santificados sejam o vosso nome.
  [00:00:05,540]  Vem a nós o vosso reino.
  [00:00:07,800]  Seja feita a vossa vontade, assim na Terra como no céu.
  [00:00:12,940]  O pão nosso de cada dia nos dá hoje.


In [31]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 3 — Correção PT com Groq                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import time, re
from datetime import datetime

ref_path = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
print("═" * 60)
if ref_path.exists():
    from srt_utils import ler_srt as _ler
    ref_legs = _ler(ref_path)
    print(f"✅ Referência PT encontrada: {len(ref_legs)} segmentos")
else:
    print(f"⚠️  Referência PT não encontrada — corrigindo sem referência")
print("═" * 60)

def corrigir_com_retry(max_tentativas=5):
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"\n🔄 Tentativa {tentativa}/{max_tentativas} - {datetime.now().strftime('%H:%M:%S')}")
            return pipeline.fase3_corrigir_pt()
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate limit' in msg.lower():
                m = re.search(r'(\d+)m(\d+\.?\d*)s', msg)
                espera = int(m.group(1)) * 60 + float(m.group(2)) if m else 30
                print(f"\n⏳ Rate limit — aguardando {espera:.0f}s...")
                for i in range(int(espera), 0, -5):
                    print(f"   ⏰ {i}s...", end='\r')
                    time.sleep(5)
                continue
            raise
    raise Exception(f"Falha após {max_tentativas} tentativas")

try:
    legendas_pt = corrigir_com_retry()
    print(f"\n✅ {len(legendas_pt)} legendas PT corrigidas")
    for leg in legendas_pt:
        print(f'  #{leg.id:02d}  {leg.texto}')
except Exception as e:
    print(f'\n❌ Erro na correção: {e}')

# ── FASE 4: Traduções EN/ES/FR ───────────────────────────────────────────────
from srt_utils import ler_srt
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)

pipeline.legendas_idiomas = pipeline.fase4_traduzir(legendas_pt)

print('Primeiras 2 legendas por idioma:')
for i in range(min(2, len(legendas_pt))):
    for lang in config.IDIOMAS:
        legs = pipeline.legendas_idiomas.get(lang, [])
        if i < len(legs):
            print(f'  {lang.upper()}: {legs[i].texto}')
    print()

════════════════════════════════════════════════════════════
✅ Referência PT encontrada: 11 segmentos
════════════════════════════════════════════════════════════

🔄 Tentativa 1/5 - 11:39:22
   📺 Referência carregada: pai_nosso_yt_ref_pt.srt (11 segmentos)
   📺 Texto de referência: Pai nosso que estais no  céu, santificado seja o vosso nome, venha a nós o vosso reino, seja feita a vossa vontade, assim na terra como no céu. O pão nosso de cada dia nos dai hoje. Perdoai as nossas ...
   Corrigindo legenda 1/7: 'Pai nosso que estáis no céu, santificados sejam o ...'
   Corrigindo legenda 2/7: 'Vem a nós o vosso reino....'
   Corrigindo legenda 3/7: 'Seja feita a vossa vontade, assim na Terra como no...'
   Corrigindo legenda 4/7: 'O pão nosso de cada dia nos dá hoje....'
   Corrigindo legenda 5/7: 'Perdua aí as nossas ofensas....'
   Corrigindo legenda 6/7: 'Assim como nós perduamos aqui nos tem ofendido....'
   Corrigindo legenda 7/7: 'E não nos deixe cair em tentação, mas livrar-nos d..

### ▶ Fase 5A — Classificação morfológica (Groq) + Pacote de revisão

O pipeline vai:
1. Classificar todos os idiomas via Groq (forçando do zero)
2. Salvar os JSONs brutos em `brutos/` no Drive
3. **Gerar o pacote de revisão** completo em `correcoes/` no Drive:
   - `prompt_revisao.md` — instruções para a IA revisora
   - `relatorio_classificacoes.csv` — todas as palavras com coluna `Sugestao`
   - Os 4 JSONs brutos prontos para correção
4. **Pausar** e mostrar instruções para a Fase 5B

In [34]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5A — Classificação Groq COM CACHE (retoma de onde parou)  ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt
import time
import re
import json
from pathlib import Path
from datetime import datetime

if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)
if not pipeline.legendas_idiomas:
    pipeline.legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)

print("═" * 70)
print("🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA COM CACHE")
print("═" * 70)

# Configurações de delay
DELAY_ENTRE_LEGENDAS = 5.0      # 5 segundos
DELAY_ENTRE_IDIOMAS = 20.0      # 20 segundos
DELAY_RATE_LIMIT = 90           # 1.5 minuto

# Pastas de cache
pasta_brutos = Path(f'/content/drive/MyDrive/pipeline/brutos/{config.NOME_ORACAO}')
pasta_brutos.mkdir(parents=True, exist_ok=True)
pasta_cache_local = Path('/content')

def verificar_cache(lang):
    """Verifica se já existe JSON no Drive ou cache local"""
    # 1º Verifica no Drive (brutos)
    drive_json = pasta_brutos / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
    if drive_json.exists():
        return drive_json, 'DRIVE (brutos/)'

    # 2º Verifica no cache local
    cache_json = pasta_cache_local / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
    if cache_json.exists():
        return cache_json, 'CACHE LOCAL'

    return None, None

def carregar_do_cache(lang, legendas):
    """Carrega classificações do cache se existir"""
    json_path, origem = verificar_cache(lang)
    if json_path:
        print(f"   📁 Carregando {lang.upper()} do {origem}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            dados = json.load(f)

        for leg in legendas:
            leg_id = str(leg.id)
            if leg_id in dados:
                palavras_data = dados[leg_id].get('palavras', [])
                from models import Palavra
                leg.palavras = [Palavra(texto=p['texto'], classe=p['classe']) for p in palavras_data]
        return True
    return False

def salvar_no_cache(lang, legendas):
    """Salva classificações no Drive e cache local"""
    dados = {}
    for leg in legendas:
        dados[str(leg.id)] = {
            'inicio': leg.inicio_str,
            'fim': leg.fim_str,
            'texto_original': leg.texto,
            'palavras': [{'texto': p.texto, 'classe': p.classe} for p in leg.palavras]
        }

    # Salvar no Drive
    drive_path = pasta_brutos / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
    with open(drive_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)

    # Salvar no cache local
    cache_path = pasta_cache_local / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)

    print(f"   💾 {lang.upper()} salvo no cache (Drive + local)")

# ═══════════════════════════════════════════════════════════════════
# CLASSIFICAR CADA IDIOMA (com cache)
# ═══════════════════════════════════════════════════════════════════

for i, lang in enumerate(config.IDIOMAS):
    print(f"\n{'='*50}")
    print(f"🔤 {lang.upper()} - {len(pipeline.legendas_idiomas[lang])} legendas")
    print(f"{'='*50}")

    # Tenta carregar do cache primeiro
    if carregar_do_cache(lang, pipeline.legendas_idiomas[lang]):
        print(f"   ✅ {lang.upper()} carregado do cache! (não chamou Groq)")
    else:
        print(f"   🚀 {lang.upper()} não está em cache. Classificando via Groq...")

        # Classificar cada legenda
        for j, leg in enumerate(pipeline.legendas_idiomas[lang]):
            print(f"   [{j+1}/{len(pipeline.legendas_idiomas[lang])}] {leg.texto[:50]}...", end=' ', flush=True)

            # Tentar classificar com retry
            for tentativa in range(1, 4):
                try:
                    leg.palavras = pipeline._groq.classificar_legenda(leg.texto, lang)
                    print("✅")
                    break
                except Exception as e:
                    erro_msg = str(e)
                    if '429' in erro_msg or 'rate limit' in erro_msg.lower():
                        match = re.search(r'(\d+)m(\d+\.?\d*)s', erro_msg)
                        if match:
                            minutos = int(match.group(1))
                            segundos = float(match.group(2))
                            espera = minutos * 60 + segundos
                        else:
                            espera = DELAY_RATE_LIMIT

                        print(f"⏳ Rate limit! Aguardando {espera:.0f}s...")
                        for k in range(int(espera), 0, -10):
                            print(f"      ⏰ {k}s restantes...", end='\r')
                            time.sleep(10)
                        print()
                        continue
                    else:
                        print(f"❌ {erro_msg[:50]}")
                        break

            # Salvar progresso a cada legenda (em caso de interrupção)
            salvar_no_cache(lang, pipeline.legendas_idiomas[lang][:j+1])

            # Delay entre legendas
            if j < len(pipeline.legendas_idiomas[lang]) - 1:
                print(f"   ⏳ Aguardando {DELAY_ENTRE_LEGENDAS}s...")
                time.sleep(DELAY_ENTRE_LEGENDAS)

        # Salvar cache completo do idioma
        salvar_no_cache(lang, pipeline.legendas_idiomas[lang])
        print(f"   ✅ {lang.upper()} classificado e salvo no cache!")

    # Delay entre idiomas
    if i < len(config.IDIOMAS) - 1:
        print(f"\n⏳ Aguardando {DELAY_ENTRE_IDIOMAS}s antes do próximo idioma...")
        for k in range(int(DELAY_ENTRE_IDIOMAS), 0, -5):
            print(f"   ⏰ {k}s restantes...", end='\r')
            time.sleep(5)
        print()

print("\n" + "═" * 70)
print("🎉 CLASSIFICAÇÃO CONCLUÍDA!")
print("═" * 70)

# ═══════════════════════════════════════════════════════════════════
# GERAR PACOTE DE REVISÃO
# ═══════════════════════════════════════════════════════════════════

print("\n📦 GERANDO PACOTE DE REVISÃO...")
pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)

print(f"\n📁 Pasta: MyDrive/pipeline/correcoes/{config.NOME_ORACAO}/")
print("\n📋 PRÓXIMOS PASSOS:")
print("   1. Copie o prompt_revisao.md e cole na IA")
print("   2. Anexe os 4 JSONs")
print("   3. Salve os JSONs corrigidos na mesma pasta")
print("   4. Execute a FASE 5C")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA COM CACHE
══════════════════════════════════════════════════════════════════════

🔤 PT - 7 legendas
   📁 Carregando PT do DRIVE (brutos/)...
   ✅ PT carregado do cache! (não chamou Groq)

⏳ Aguardando 20.0s antes do próximo idioma...


🔤 EN - 7 legendas
   📁 Carregando EN do DRIVE (brutos/)...
   ✅ EN carregado do cache! (não chamou Groq)

⏳ Aguardando 20.0s antes do próximo idioma...


🔤 ES - 7 legendas
   📁 Carregando ES do DRIVE (brutos/)...
   ✅ ES carregado do cache! (não chamou Groq)

⏳ Aguardando 20.0s antes do próximo idioma...


🔤 FR - 7 legendas
   📁 Carregando FR do DRIVE (brutos/)...
   ✅ FR carregado do cache! (não chamou Groq)

══════════════════════════════════════════════════════════════════════
🎉 CLASSIFICAÇÃO CONCLUÍDA!
══════════════════════════════════════════════════════════════════════

📦 GERANDO PACOTE DE REVISÃO...

📁 Pasta: MyDrive/pipeline/correcoes/pai

### ⏸️ Fase 5B — Revisão manual pela IA ← **Você faz isso**

Após a Fase 5A concluir:

1. **Abra o Google Drive** → `MyDrive/pipeline/correcoes/pai_nosso/`
2. Você verá:
   - `prompt_revisao.md` ← **copie e cole numa IA junto com os 4 JSONs**
   - `relatorio_classificacoes.csv` ← consulte a coluna `Sugestao` para ver correções sugeridas
   - Os 4 JSONs brutos para corrigir
3. A IA retorna os JSONs corrigidos
4. **Salve os JSONs corrigidos na mesma pasta** (substituindo os originais)
5. Execute a Fase 5C abaixo

---
### ▶ Fase 5C — Recarregar classificações corrigidas

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5C — Recarregar do Drive após revisão                    ║
# ║  Execute APÓS salvar os JSONs corrigidos no Drive              ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📥 FASE 5C — RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS")
print("═" * 65)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')

if not pasta_correcoes.exists():
    print(f"❌ Pasta não encontrada: {pasta_correcoes}")
    print("   Execute a FASE 5A primeiro.")
else:
    jsons_ok = []
    jsons_faltando = []
    for lang in config.IDIOMAS:
        p = pasta_correcoes / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
        if p.exists():
            jsons_ok.append(lang.upper())
            print(f"   ✅ {p.name}")
        else:
            jsons_faltando.append(lang.upper())
            print(f"   ❌ {p.name}")

    if len(jsons_ok) == 4:
        print("\n✅ Todos os 4 JSONs encontrados! Carregando...")
        pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

        print("\n📋 Preview das classificações:")
        print("─" * 55)
        for lang in config.IDIOMAS:
            legs = pipeline.legendas_idiomas.get(lang, [])
            if legs and legs[0].palavras:
                leg = legs[0]
                print(f'\n  {lang.upper()} — "{leg.texto[:50]}"')
                from constants import CORES_HTML
                for p in leg.palavras[:4]:
                    ok  = '✅' if p.classe in CORES_HTML else '❌'
                    print(f'    {ok} {p.texto:<18} {p.classe}')
                if len(leg.palavras) > 4:
                    print(f'       ... (+{len(leg.palavras)-4} palavras)')

        print("\n" + "═" * 65)
        print("🎬 Pronto — execute as Fases 6, 7 e 8")
        print("═" * 65)
    else:
        print(f"\n⚠️  Faltam: {', '.join(jsons_faltando)}")
        print("   Coloque os JSONs corrigidos na pasta e execute novamente.")

### ▶ Fases 6–8 — Vídeo final (rode após a Fase 5C)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASES 6, 7 e 8 — Vídeo final                 ║
# ╚══════════════════════════════════════════════════╝

video_final = pipeline.continuar()

if video_final:
    print(f'\n🎬 Vídeo final: {video_final}')
    print(f'   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB')
    print(f'   Salvo no Drive: pasta Vídeos')

### 🚀 RUN ALL — Pipeline completo com checkpoint

Use quando quiser rodar tudo de uma vez. O pipeline pausa automaticamente na Fase 5A
se houver classificações a revisar. Após a revisão, rode a célula da Fase 5C
e depois a célula das Fases 6–8.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  RUN ALL                                       ║
# ╚══════════════════════════════════════════════════╝
from video_pipeline import ClassificacaoPendenteError

resultado = pipeline.run(
    # from_phase='clipes_cortados'  # descomente para retomar de uma fase
)

if resultado is None:
    print('\n⏸️  Pipeline pausado na Fase 5A.')
    print('   Siga as instruções acima, depois execute a célula FASE 5C.')
else:
    print(f'\n🎉 Vídeo final: {resultado}')
    print(pipeline._cp.resumo())

### 🔧 Utilitários

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  UTILITÁRIOS — descomente o que precisar       ║
# ╚══════════════════════════════════════════════════╝
from checkpoint import Checkpoint
cp = Checkpoint()
print(cp.resumo())
print()
pipeline._clf.imprimir_status()

# ── Checkpoint ────────────────────────────────────────────────────────────────
# cp.resetar_tudo()
# cp.reiniciar_de('classificacoes_feitas')

# ── Classificações ────────────────────────────────────────────────────────────
# pipeline._clf.imprimir_status()
# pipeline._clf.classificar_idioma(pipeline.legendas_idiomas['en'], 'en', forcar=True)

# Gerar pacote de revisão manualmente (sem reclassificar):
# pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)
# print(f"📦 Pacote: {pacote}")

# Verificar classes inválidas:
# from constants import CORES_HTML
# for lang, legendas in pipeline.legendas_idiomas.items():
#     for leg in legendas:
#         for p in leg.palavras:
#             if p.classe not in CORES_HTML:
#                 print(f'  [{lang.upper()} leg.{leg.id}] "{p.texto}" → {p.classe} ❌')